# OffScript — Phase 2: Baseline Pitch Selection Model

In [ ]:
# notebooks/05_baseline_model.ipynb

import sys
sys.path.append('../src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import xgboost as xgb

from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from pitch_analysis import load_clean_data

# Load clean data
data = load_clean_data()
print(f"Modeling dataset: {len(data)} pitches")
print(f"Pitch types: {sorted(data['pitch_type'].unique())}")

## Step 1 — Add Pitcher Identity as Feature

In [ ]:
# Encode pitcher name as a numeric feature
pitcher_encoder = LabelEncoder()
data['pitcher_encoded'] = pitcher_encoder.fit_transform(data['pitcher_name'])

joblib.dump(pitcher_encoder, '../models/pitcher_encoder.pkl')

print("=== Pitcher Encoding ===")
for name, code in zip(pitcher_encoder.classes_,
                       range(len(pitcher_encoder.classes_))):
    print(f"{name}: {code}")

## Step 2 — Define Feature Matrix

In [ ]:
feature_cols = [
    'balls', 'strikes', 'inning', 'score_diff',
    'on_1b', 'on_2b', 'on_3b',
    'runners_on', 'scoring_position',
    'stand_encoded', 'pitcher_encoded', 'count_leverage'
]

# Engineer features
data['stand_encoded'] = (data['stand'] == 'R').astype(int)
data['runners_on'] = (data['on_1b'].fillna(0) +
                      data['on_2b'].fillna(0) +
                      data['on_3b'].fillna(0))
data['scoring_position'] = (
    (data['on_2b'].fillna(0) + data['on_3b'].fillna(0)) > 0
).astype(int)
data['count_leverage'] = (
    (data['strikes'] == 2).astype(int) * 2 +
    (data['balls'] == 3).astype(int) * 2 +
    (data['strikes'] == 1).astype(int) +
    (data['balls'] == 2).astype(int)
)

print("=== Feature Matrix Preview ===")
print(data[feature_cols].describe())
print(f"\nTarget classes: {sorted(data['pitch_type'].unique())}")

## Step 3 — Train/Test Split

In [ ]:
X = data[feature_cols].fillna(0)
le = LabelEncoder()
y = le.fit_transform(data['pitch_type'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print("\n=== Class Distribution in Test Set ===")
print(pd.Series(le.inverse_transform(y_test)).value_counts())

## Step 4 — Calculate Class Weights

In [ ]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Cap between 0.5 and 2.0 to prevent overcorrection
class_weights = np.clip(class_weights, 0.5, 2.0)
sample_weights = class_weights[y_train]

print("=== Adjusted Class Weights ===")
for cls, weight in zip(le.classes_, class_weights):
    count = (y_train == le.transform([cls])[0]).sum()
    print(f"{cls}: count={count}, weight={weight:.3f}")

## Step 5 — Train XGBoost Model

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

print(f"Baseline Model Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## A Note on Model Accuracy

Pitch selection involves intentional deception and randomness by design.
A pitcher who is perfectly predictable would be ineffective at the major
league level. An accuracy meaningfully above random chance (~11% for 9 
classes) demonstrates the model has learned real patterns. The goal of 
this model is not perfect prediction but rather establishing a statistical
baseline from which individual pitcher deviations can be measured.

## Step 6 — Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Feature Importance ===")
print(importance_df)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature',
            hue='feature', legend=False, palette='viridis', ax=ax)
ax.set_title("Baseline Model — Feature Importance", fontsize=14)
ax.set_xlabel("Importance Score")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None] * 100

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_pct, annot=True, fmt='.1f',
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap='Blues', ax=ax)
ax.set_title("Confusion Matrix — Pitch Type Prediction (%)", fontsize=14)
ax.set_ylabel("Actual Pitch")
ax.set_xlabel("Predicted Pitch")
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — SHAP Explainability

In [ ]:
print("Generating SHAP values — this may take a moment...")
explainer = shap.TreeExplainer(model)
shap_sample = X_test.sample(2000, random_state=42)
shap_values = explainer.shap_values(shap_sample)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, shap_sample,
                  feature_names=feature_cols,
                  class_names=le.classes_,
                  show=False)
plt.title("SHAP Summary — What Drives Pitch Selection?")
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Save Models

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(model, '../models/baseline_pitch_model.pkl')
joblib.dump(le, '../models/label_encoder.pkl')
joblib.dump(pitcher_encoder, '../models/pitcher_encoder.pkl')

print("=== Models Saved ===")
print("baseline_pitch_model.pkl")
print("label_encoder.pkl")
print("pitcher_encoder.pkl")

## Step 10 — Phase 2 Summary Statistics

In [ ]:
print(f"Model last run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("\n=== PHASE 2 SUMMARY STATISTICS ===")
print(f"\nDataset:")
print(f"  Total pitches: {len(data)}")
print(f"  Training samples: {len(X_train)}")
print(f"  Testing samples: {len(X_test)}")
print(f"  Pitch types modeled: {sorted(le.classes_)}")

print(f"\nModel Performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"  Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")

report = classification_report(y_test, y_pred,
                                target_names=le.classes_,
                                output_dict=True)
f1_scores = {k: v['f1-score'] for k, v in report.items()
             if k in le.classes_}

print(f"\nF1 Scores by Pitch Type:")
for pitch, f1 in sorted(f1_scores.items(),
                         key=lambda x: x[1], reverse=True):
    print(f"  {pitch}: {f1:.3f}")

print(f"\nFeature Importance (top 3):")
for _, row in importance_df.head(3).iterrows():
    print(f"  {row['feature']}: {row['importance']:.3f} "
          f"({row['importance']*100:.1f}%)")

## Phase 2 Results Summary

### Dataset
- **Total pitches modeled:** 72,098 across 15 pitchers (2023-2024)
- **Training samples:** 57,678
- **Testing samples:** 14,420
- **Pitch types:** CH, CU, FC, FF, FS, KC, SI, SL, ST

### Model Performance
- **Accuracy:** 0.424
- **Balanced Accuracy:** 0.482
- **Strongest classes:** SI (0.525), FC (0.502), SL (0.468)
- **Most challenging classes:** KC (0.248), CU (0.299)

### Feature Importance Findings
- **Pitcher identity** is the dominant predictor at 59.8%, confirming
  that individual arsenal and tendencies drive pitch selection more
  than any situational factor
- **Batter handedness** is the strongest situational feature at 13.4%,
  consistent with conventional pitching wisdom
- **Strike count** (5.4%) and **ball count** (3.8%) are the strongest
  pure count features, reflecting how two-strike and three-ball counts
  shift pitch selection

### Interpretation
This baseline model establishes the statistical foundation for deviation
analysis in Phase 3. The dominance of pitcher identity as a feature
raises an important design question for deviation scoring — deviations
can be measured against universal league norms (removing pitcher identity)
or against each pitcher's own tendencies (keeping pitcher identity).
Phase 3 will explore both approaches.

### Model Limitations
- Rare pitch types (KC, ST) remain difficult to predict precisely
  due to limited sample sizes
- Intentional deception is inherent to pitching, creating a natural
  ceiling on predictability
- A balanced accuracy of 0.482 against a random baseline of ~0.111
  for 9 classes represents meaningful signal